# Unitree G1 — Cross-Seed Replication 用 学習パイプライン(整理版)「前進のみ」の指令seed1-8

このノートブックは、**同一条件・異なる乱数seedで、8個のG1歩行policyを作成する**ための、
整理済みの一直線パイプラインです。

**上から順番に実行してください。** 途中経過の確認や、デバッグ用の探索セルは、
すべて取り除いてあります(必要な知見は、すでにコードの中に反映済みです)。

構成:
1. セットアップ(インストール・GPU確認)
2. 設定(タスク・8つのseed・向きの固定など)
3. 学習(8seed分、チャンク分割+Google Driveへの自動退避つき)
4. 向きの確認(8seedすべてで、意図した方向を向いているかを検証)
5. 動画の生成(8seed分)


## 1. セットアップ

In [ ]:
import os
if os.path.basename(os.getcwd()) == "notebooks":   # リポジトリ内で開いた場合はリポジトリ直下で作業
    os.chdir("..")
os.environ["MUJOCO_GL"] = "egl"        # GPU描画バックエンド(学習・動画の両方で、一貫してeglを使う)
os.environ["PYOPENGL_PLATFORM"] = ""   # eglとの衝突を避けるため、明示的に空にしておく
os.environ["WANDB_MODE"] = "offline"   # 実験管理サービス(wandb)へのログインを省略

!nvidia-smi -L || echo '⚠️ GPU が必要です(Colab: ランタイム → ランタイムのタイプを変更 → GPU)'

# バージョン整合(重要)。最新版同士は相互に非互換なので、検証済みの組合せに固定する
%pip install -q "mjlab==1.2.0" "mujoco==3.5.0" "warp-lang==1.12.1" "numpy>=2.0,<2.3" scipy mediapy

# pip が numpy を入れ替えた場合、カーネルには旧 numpy が読み込まれたまま残ることがある。
# メモリとディスクで版がずれたまま進むと後段の import が壊れるので、ここで検知して止める
import importlib.metadata, numpy
_numpy_on_disk = importlib.metadata.version("numpy")
if numpy.__version__ != _numpy_on_disk:
    raise RuntimeError(
        f"numpy の入替を完了するため、ランタイムの再起動が必要です(メモリ: {numpy.__version__} / ディスク: {_numpy_on_disk})。"
        "メニューの「ランタイム → セッションを再起動」の後、もう一度上から実行してください(2回目のinstallは数秒で終わります)"
    )

# GPUの無いランタイムならここで止める
import shutil, subprocess, sys
if shutil.which("nvidia-smi") is None or subprocess.run(["nvidia-smi", "-L"], capture_output=True).returncode != 0:
    raise RuntimeError("GPU ランタイムに切り替えてください(ランタイム → ランタイムのタイプを変更 → GPU)")

import torch
if not torch.cuda.is_available():
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "torch==2.6.0",
                    "--index-url", "https://download.pytorch.org/whl/cu124"], check=True)
    print("↺ torch を cu124 に入替えました。ランタイム再起動→上から再実行してください。")

assert torch.cuda.is_available(), "CUDA GPU が見つかりません"
print("✅ install done:", torch.cuda.get_device_name(0))


## 2. 設定(ここだけ書き換えれば、全体に反映されます)

In [ ]:
import math

# --- タスク・GPU設定 ---
TASK   = "Mjlab-Velocity-Flat-Unitree-G1"
DEVICE = "cuda:0"
_vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
NUM_ENVS = 4096 if _vram_gb >= 20 else 2048   # 並列体数。24GB級(3090/A100)は4096、T4/L4(16GB)は2048

# --- cross-seed replication 用の設定 ---
AGENT_SEEDS = [1, 2, 3, 4, 5, 6, 7, 8]      # 8個のpolicyそれぞれに割り当てる、学習(PPO探索)側のseed
ENV_SEED = 100                     # 環境側のseedは、8個すべてで固定
                                    # (「学習過程の違いだけ」を見るためのcross-seed replicationの設計)
RIGHT_FACING_YAW = "(-1.5708,-1.5708)"  # 初期姿勢を右向き(-90度)に固定するコマンドライン引数の値
RIGHT_FACING_YAW_RAD = -math.pi / 2     # 同じ値を、Python側の数値としても使う

# --- 学習量の設定 ---
TOTAL_ITERATIONS = 3000   # 最終的に到達させたい、合計のiteration数
CHUNK_SIZE = 500          # 1回あたりの学習量(この単位でGoogle Driveへ退避する)
                          # ★ まず TOTAL_ITERATIONS=500 程度の小さい値で、パイプライン全体が
                          #    最後まで正常に動くことを確認してから、本番の値に戻すことを推奨します

print(f"GPU: {torch.cuda.get_device_name(0)} ({_vram_gb:.0f} GB) → num_envs={NUM_ENVS}")
print(f"8個のpolicy、それぞれの学習seed: {AGENT_SEEDS}(環境seedは全て{ENV_SEED}で固定)")
print(f"1個あたり、合計{TOTAL_ITERATIONS} iterations({CHUNK_SIZE}ずつに分割して学習)")


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_BACKUP_DIR = "/content/drive/MyDrive/g1_cross_seed_logs"
os.makedirs(DRIVE_BACKUP_DIR, exist_ok=True)
print(f"退避先: {DRIVE_BACKUP_DIR}")

# --- もし、以前の学習結果がDriveに残っていれば、ここで復元できます ---
# (不要な場合は、このセルは実行しなくて構いません)
RESTORE_FROM_DRIVE = False  # 復元したい場合は True に変更してから実行
if RESTORE_FROM_DRIVE:
    !cp -r "{DRIVE_BACKUP_DIR}/logs" /content/ 2>/dev/null || echo "Drive側に、復元できるlogsが見つかりませんでした"


## 3. 学習(8seed分、resumeなし・単一の連続実行+定期的なDrive退避)

**この処理は、時間がかかります**(`TOTAL_ITERATIONS`と`NUM_ENVS`次第です)。

> **設計変更の経緯**: 当初は「学習をチャンクに分割し、`--agent.resume`で再開する」
> 設計でしたが、実験の結果、`--agent.resume`には次の問題があることが分かりました。
> - 指定した目標iteration数を超えて、予測できない量だけ進んでしまう
>   (1000を指定して1250、さらに1350まで進む、等)
> - 実行のたびに、新しい日時フォルダが作られてしまう
>
> これは、**cross-seed replicationの前提(8seed間で学習条件を厳密に揃える)を
> 損なう**ため、**resumeは使わないことにしました**。
>
> 代わりに、**学習プロセスを1回も止めずに、最初から最後まで連続して実行**し、
> その裏側で、**一定時間ごとに`logs/`フォルダをGoogle Driveへコピーし続ける**、
> という設計に変更しています。これにより、
> - 8seedとも、`TOTAL_ITERATIONS`に正確に到達する(途中で余計に進まない)
> - それでいて、クラッシュ対策(こまめなDriveへの退避)も維持される
>
> という、両方が満たされます。なお、mjlab自体も`--agent.save-interval`
> (既定50 iteration)ごとに、ローカルへcheckpointを刻み続けているため、
> 今回の変更は「その刻みを、Driveにも定期的に反映させる」という位置づけです。


In [ ]:
#学習
import subprocess
import glob
import re
import time
from pathlib import Path

BACKUP_INTERVAL_SEC = 600   # 10分ごとに、学習中でもDriveへ退避する


def backup_to_drive():
    """logs/ フォルダ全体を、Google Driveへコピーする"""
    subprocess.run(["cp", "-r", "logs", DRIVE_BACKUP_DIR], check=False)
    print(f"  [退避完了] {DRIVE_BACKUP_DIR} へコピーしました")


def get_latest_checkpoint_iteration(run_name):
    """指定したrun_nameに一致する、最新(日付が一番新しい)フォルダの、
    最新checkpoint番号を返す。resumeを使わない設計では、通常フォルダは
    1seedにつき1つだけになる。"""
    run_dirs = sorted(glob.glob(f"logs/rsl_rl/g1_velocity/*_{run_name}"))
    if not run_dirs:
        return None
    latest_dir = run_dirs[-1]
    ckpts = sorted(Path(latest_dir).glob("model_*.pt"),
                    key=lambda p: int(re.search(r"model_(\d+)", p.name).group(1)))
    if not ckpts:
        return None
    return int(re.search(r"model_(\d+)", ckpts[-1].name).group(1))


def run_training_with_periodic_backup(seed, total_iterations, backup_interval_sec=BACKUP_INTERVAL_SEC):
    """学習を1回も止めずに(resumeなし)実行しつつ、裏で定期的にDriveへ退避する"""
    run_name = f"seed{seed}"
    cmd = [
        sys.executable, "-m", "mjlab.scripts.train", TASK,
        f"--env.scene.num-envs={NUM_ENVS}",
        f"--agent.max-iterations={total_iterations}",
        f"--env.events.reset-base.params.pose-range.yaw={RIGHT_FACING_YAW}",
        f"--env.seed={ENV_SEED}",
        f"--agent.seed={seed}",
        "--agent.logger=tensorboard",
        f"--agent.run-name={run_name}",
    ]
    print(f"  実行コマンド: {' '.join(cmd)}")
    proc = subprocess.Popen(cmd)   # runではなくPopen: 裏側で学習を進めながら、退避を並行させる

    while proc.poll() is None:     # 学習が、まだ終わっていない間、一定間隔で退避を繰り返す
        time.sleep(backup_interval_sec)
        if proc.poll() is None:    # 待っている間に終わっていたら、退避のログは出さない
            backup_to_drive()
            print(f"  [経過中の退避] seed{seed} はまだ実行中です")

    backup_to_drive()   # 完了時にも、念のためもう一度、最終状態を退避する
    return proc.returncode == 0


# --- 本番学習のメインループ(resumeなし、seedごとに単一の連続実行) ---
final_iterations = {}
for seed in AGENT_SEEDS:
    print(f"\n{'='*60}\n=== seed{seed} の学習を開始します(目標: {TOTAL_ITERATIONS} iterations、resumeなし) ===\n{'='*60}")

    success = run_training_with_periodic_backup(seed, TOTAL_ITERATIONS)
    if not success:
        print(f"  [警告] seed{seed} の学習が、正常終了しなかった可能性があります(returncode異常)")

    final = get_latest_checkpoint_iteration(f"seed{seed}")
    final_iterations[seed] = final
    print(f"=== seed{seed} が完了しました(最終到達: {final} iterations) ===\n")

print("\n全seedの学習が完了しました。")
print("各seedの最終到達点:", final_iterations)


## 4. 向きの確認(8seedすべてで、意図した方向を向いているか)

**注意**: 動画にした際、カメラの仕様上、見た目は正面向きのままに見えることがあります。
これは正常です。ロボットの実際の物理的な向きは、ここでの角度の数値で確認してください。


In [ ]:
import glob, re
from dataclasses import asdict
from pathlib import Path

import numpy as np
from mjlab.envs import ManagerBasedRlEnv
from mjlab.rl import RslRlVecEnvWrapper
from mjlab.tasks.registry import load_env_cfg, load_rl_cfg

for seed in AGENT_SEEDS:
    run_dirs = sorted(glob.glob(f"logs/rsl_rl/g1_velocity/*_seed{seed}"))
    if not run_dirs:
        print(f"seed{seed}: runディレクトリが見つかりません")
        continue
    RUN_DIR = run_dirs[-1]

    play_cfg = load_env_cfg(TASK, play=True)
    play_cfg.scene.num_envs = 1
    play_cfg.events["reset_base"].params["pose_range"]["yaw"] = (RIGHT_FACING_YAW_RAD, RIGHT_FACING_YAW_RAD)
    agent_cfg = load_rl_cfg(TASK)

    raw_env = ManagerBasedRlEnv(cfg=play_cfg, device=DEVICE, render_mode="rgb_array")
    env = RslRlVecEnvWrapper(raw_env, clip_actions=agent_cfg.clip_actions)
    # ↑ この行の内部で、既にreset()が1回実行済み(yaw設定もここで反映される)。
    #    これ以降、reset()は絶対に呼ばない(呼ぶとレンダラーが壊れることが分かっている)

    qpos = raw_env.sim.data.qpos[0].cpu().numpy()
    quat = qpos[3:7]
    w, x, y, z = quat
    yaw_rad = np.arctan2(2*(w*z + x*y), 1 - 2*(y*y + z*z))
    yaw_deg = np.degrees(yaw_rad)
    expected_deg = np.degrees(RIGHT_FACING_YAW_RAD)
    ok = "✅" if abs(yaw_deg - expected_deg) < 1.0 else "❌"
    print(f"{ok} seed{seed}: 実際のyaw角度 = {yaw_deg:.1f}度(設定値: {expected_deg:.1f}度)")

    env.close()
    del env, raw_env
    torch.cuda.empty_cache()


## 5. 動画の生成(8seed分)

In [ ]:
#Mydriveから復元
from google.colab import drive
drive.mount('/content/drive')

!cp -r "/content/drive/MyDrive/g1_cross_seed_logs/logs" /content/ && echo "復元しました" || echo "復元に失敗しました。パスを確認してください"

In [ ]:
#logsフォルダseed確認
import os
print("logsフォルダが存在するか:", os.path.exists("logs"))
if os.path.exists("logs/rsl_rl/g1_velocity"):
    print("中身:", os.listdir("logs/rsl_rl/g1_velocity"))

In [ ]:
#動画作成　前進指令のみ
import mediapy as media
import glob
import re
import math
import torch
import numpy as np
from pathlib import Path
from dataclasses import asdict
from mjlab.envs import ManagerBasedRlEnv
from mjlab.rl import MjlabOnPolicyRunner, RslRlVecEnvWrapper
from mjlab.tasks.registry import load_env_cfg, load_rl_cfg

TASK = "Mjlab-Velocity-Flat-Unitree-G1"
DEVICE = "cuda:0"
AGENT_SEEDS = [1, 2, 3, 4, 5, 6, 7, 8]
RIGHT_FACING_YAW_RAD = -math.pi / 2

Path("videos").mkdir(exist_ok=True)

for seed in AGENT_SEEDS:
    print(f"=== seed{seed} の動画を作成中 ===")

    run_dirs = sorted(glob.glob(f"logs/rsl_rl/g1_velocity/*_seed{seed}"))
    if not run_dirs:
        print(f"  seed{seed} のrunディレクトリが見つかりません。スキップします。")
        continue
    RUN_DIR = run_dirs[-1]

    ckpt = max(Path(RUN_DIR).glob("model_*.pt"),
               key=lambda p: int(re.search(r"model_(\d+)", p.name).group(1)))
    print(f"  checkpoint: {ckpt}")

    play_cfg = load_env_cfg(TASK, play=True)
    play_cfg.scene.num_envs = 1
    play_cfg.events["reset_base"].params["pose_range"]["yaw"] = (RIGHT_FACING_YAW_RAD, RIGHT_FACING_YAW_RAD)
    agent_cfg = load_rl_cfg(TASK)

    raw_env = ManagerBasedRlEnv(cfg=play_cfg, device=DEVICE, render_mode="rgb_array")
    env = RslRlVecEnvWrapper(raw_env, clip_actions=agent_cfg.clip_actions)
    # ↑ この時点で、内部でreset()が1回実行済み。以降reset()は呼ばない

    runner = MjlabOnPolicyRunner(env, asdict(agent_cfg), device=DEVICE)
    runner.load(str(ckpt), load_cfg={"actor": True}, strict=True, map_location=DEVICE)
    policy = runner.get_inference_policy(device=DEVICE)

    # --- 8seed全て、公平に比較できるよう、指令(コマンド)を「まっすぐ前進」に固定する ---
    #
    # 重要な注意(実験で確認済み): ループの外で1回だけ設定するのでは不十分。
    # このタスクは heading_command=True (既定) が有効で、reset()時にランダムに決まる
    # "目標heading" へ向かうための回転速度が、毎ステップ、環境の内部で自動計算され、
    # 一度だけ設定した「回転なし」の指令を、裏で上書きし続けてしまう。
    # (症状: 最初は正面を向いて歩き出すが、次第に左右どちらかへ、じわじわ回転していく)
    #
    # そのため、以下では「ループの外で1回」ではなく、「ループの中で、毎ステップ」、
    # 指令を強制的に上書きする。これにより、環境側の内部ロジックが何を計算していようと、
    # 最終的に必ず「前進のみ」の指令が使われることを保証する。
    FORWARD_ONLY_COMMAND = torch.tensor([[1.0, 0.0, 0.0]], device=DEVICE)  # 前進1.0 m/s、横移動・回転なし

    obs = env.get_observations()
    frames = []
    for _ in range(150):  # 150 step ≈ 3秒
        raw_env.command_manager.get_term("twist").vel_command_b[:] = FORWARD_ONLY_COMMAND
        with torch.no_grad():
            action = policy(obs)
        obs, _, _, _ = env.step(action)
        frame = np.asarray(raw_env.render())
        frames.append(frame[0] if frame.ndim == 4 else frame)

    out_path = f"videos/g1_seed{seed}.mp4"
    media.write_video(out_path, frames, fps=int(raw_env.metadata.get("render_fps", 50)))
    print(f"  保存しました: {out_path}")

    env.close()
    del runner, policy, env, raw_env
    torch.cuda.empty_cache()

print("=== 全seedの動画作成が完了しました ===")

In [ ]:
import mediapy as media
import glob
import re
import math
import torch
import numpy as np
from pathlib import Path
from dataclasses import asdict
from mjlab.envs import ManagerBasedRlEnv
from mjlab.rl import MjlabOnPolicyRunner, RslRlVecEnvWrapper
from mjlab.tasks.registry import load_env_cfg, load_rl_cfg

TASK = "Mjlab-Velocity-Flat-Unitree-G1"
DEVICE = "cuda:0"
AGENT_SEEDS = [1, 2, 3, 4, 5]
RIGHT_FACING_YAW_RAD = -math.pi / 2

Path("videos").mkdir(exist_ok=True)

for seed in AGENT_SEEDS:
    print(f"=== seed{seed} の動画を作成中 ===")

    run_dirs = sorted(glob.glob(f"logs/rsl_rl/g1_velocity/*_seed{seed}"))
    if not run_dirs:
        print(f"  seed{seed} のrunディレクトリが見つかりません。スキップします。")
        continue
    RUN_DIR = run_dirs[-1]

    ckpt = max(Path(RUN_DIR).glob("model_*.pt"),
               key=lambda p: int(re.search(r"model_(\d+)", p.name).group(1)))
    print(f"  checkpoint: {ckpt}")

    play_cfg = load_env_cfg(TASK, play=True)
    play_cfg.scene.num_envs = 1
    play_cfg.events["reset_base"].params["pose_range"]["yaw"] = (RIGHT_FACING_YAW_RAD, RIGHT_FACING_YAW_RAD)
    agent_cfg = load_rl_cfg(TASK)

    raw_env = ManagerBasedRlEnv(cfg=play_cfg, device=DEVICE, render_mode="rgb_array")
    env = RslRlVecEnvWrapper(raw_env, clip_actions=agent_cfg.clip_actions)

    runner = MjlabOnPolicyRunner(env, asdict(agent_cfg), device=DEVICE)
    runner.load(str(ckpt), load_cfg={"actor": True}, strict=True, map_location=DEVICE)
    policy = runner.get_inference_policy(device=DEVICE)

    # --- 8seed全て、公平に比較できるよう、指令(コマンド)を「まっすぐ前進」に固定する ---
    FORWARD_ONLY_COMMAND = [1.0, 0.0, 0.0]
    raw_env.command_manager.get_term("twist").vel_command_b[:] = torch.tensor(
        [FORWARD_ONLY_COMMAND], device=DEVICE
    )

    obs = env.get_observations()
    frames = []
    for _ in range(150):
        with torch.no_grad():
            action = policy(obs)
        obs, _, _, _ = env.step(action)
        frame = np.asarray(raw_env.render())
        frames.append(frame[0] if frame.ndim == 4 else frame)

    out_path = f"videos/g1_seed{seed}.mp4"
    media.write_video(out_path, frames, fps=int(raw_env.metadata.get("render_fps", 50)))
    print(f"  保存しました: {out_path}")

    env.close()
    del runner, policy, env, raw_env
    torch.cuda.empty_cache()

print("=== 全seedの動画作成が完了しました ===")

In [ ]:
from IPython.display import Video, display

for seed in AGENT_SEEDS:
    path = f"videos/g1_seed{seed}.mp4"
    if os.path.exists(path):
        print(f"--- seed{seed} ---")
        display(Video(path, embed=True, width=480))
